In [1]:
import polars as pl
from pybiomart import Server
import os
import numpy as np
import pandas as pd

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/dataset'
HISTONE_DIR = os.path.join(WORKING_DIR, 'dataset', 'histone_overlap')

# Dataset Loading

## Load Preprocessed HepG2 Data

In [3]:
# Schema for gene data
schema_genes = pl.Schema({
    "chromosome"                : pl.String(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "locus"                     : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64()
})

In [4]:
# Read the gene data
gene_pl = pl.read_csv(os.path.join(WORKING_DIR, "dataset", "ensembl_top1.csv"),
                     schema=schema_genes,
                     has_header=False,
                     separator="\t")

In [5]:
# Get the gene list
gene_list = gene_pl.select(pl.col('gene_id')).to_numpy().flatten()
len(gene_list)

22154

## Load Overlap Histone Data

In [6]:
# Schema for overlap histone
schema_histone_overlap = pl.Schema({
    "chromosome"                : pl.String(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "locus"                     : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64(),
    "histone_chr"               : pl.String(),
    "histone_start"             : pl.Int64(),
    "histone_end"               : pl.Int64(),
    "histone_name"              : pl.String()
})

## Building Matrix Data

In [7]:
def build_matrix(gene_list, histone_name):
    # Open histone overlap file
    histone_pl = pl.read_csv(os.path.join(HISTONE_DIR, f'ensembl_top1_{histone_name}.bed'), 
                         schema = schema_histone_overlap, 
                         separator="\t",
                         has_header=False)

    # Create partitions to optimize searching
    histone_partitions = histone_pl.partition_by("gene_id", as_dict=True)

    # Build histone matrix
    histone_list = []
    i = 1
    for gene in gene_list:
        histone_arr = np.zeros(4000, dtype=float)
        count = 0
        
        if((gene,) in histone_partitions):
            # print(f"{gene} -  EXIST")
            
            partition = histone_partitions[(gene,)].to_numpy()
            count = partition.shape[0]
    
            for row in partition:
                start = row[1]
                idx_start = row[17] - start
                idx_end = row[18] - start
                # print(f"{row[19]}: {idx_start} - {idx_end}")
                
                histone_arr[idx_start: idx_end] = 1.0
        # else:
        #     print(f"{gene} -  NOT EXIST")
    
        # print(f"SUM: {np.sum(histone_arr)}, COUNT: {count}")
        entry = [gene, histone_arr, count]
    
        # print(f"{i}/{len(gene_list)}: {gene}, count: {count}")
    
        histone_list.append(entry)
        i += 1

    # Convert the array into Numpy arrays
    histone_np = np.array(histone_list, dtype="object")

    # Convert to Pandas dataframe
    histone_pd = pd.DataFrame(histone_np, columns=["gene_id", f"{histone_name}", f"{histone_name}_count"])

    # Convert to Polars dataframe
    histone_pl = pl.from_pandas(histone_pd)

    return histone_pl

In [8]:
# histone_names = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']

H3K4me3_pl = build_matrix(gene_list, 'H3K4me3')
H3K9ac_pl  = build_matrix(gene_list, 'H3K9ac')
H3K9me3_pl = build_matrix(gene_list, 'H3K9me3')
H3K27ac_pl = build_matrix(gene_list, 'H3K27ac')
H3K27me3_pl = build_matrix(gene_list, 'H3K27me3')

## Join into single matrix

In [9]:
# Join into single Polars dataframe
gene_w_histone = H3K4me3_pl \
                    .join(H3K9ac_pl, on='gene_id') \
                    .join(H3K9me3_pl, on='gene_id') \
                    .join(H3K27ac_pl, on='gene_id') \
                    .join(H3K27me3_pl, on='gene_id')

In [10]:
gene_w_histone

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0
…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0


In [11]:
gene_w_histone.write_parquet(os.path.join(WORKING_DIR, 'dataset', 'gene_w_histone.parquet'))

In [12]:
test_pl = pl.read_parquet(os.path.join(WORKING_DIR, 'dataset', 'gene_w_histone.parquet'))

In [13]:
test_pl

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0
…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
